# Exchangeability Plotting Notebook
Generate publication-style plots from an exchangeability CSV.

In [ ]:
import os
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
from scripts.plot_exchangeability import _prepare, _aggregate_for_curves, _plot_metric, _plot_train_val

In [ ]:
BASE_SAVE_DIR = Path(os.environ.get('BASE_SAVE_DIR', '/n/netscratch/kempner_pehlevan_lab/Lab/ilavie/exchangeability_outputs'))
candidates = [
    BASE_SAVE_DIR / 'exchangeability_metrics_w32.csv',
    BASE_SAVE_DIR / 'exchangeability_metrics.csv',
    Path('outputs/exchangeability_metrics_w32.csv'),
    Path('outputs/exchangeability_metrics.csv'),
]
CSV_PATH = next((p for p in candidates if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError('No exchangeability metrics CSV found. Run notebooks/exchangeability_analysis.ipynb first.')
OUT_DIR = CSV_PATH.parent / f'plots_{CSV_PATH.stem}_notebook'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Using CSV_PATH={CSV_PATH}')
print(f'Writing OUT_DIR={OUT_DIR}')
df = _prepare(pd.read_csv(CSV_PATH))
curves = _aggregate_for_curves(df)

In [ ]:
fig_ks = _plot_metric(
    curves,
    metric='ks_distance',
    lo='ks_distance_p10',
    hi='ks_distance_p90',
    out_path=str(OUT_DIR / 'ks_distance_vs_images_seen.pdf'),
    title='KS Distance vs Images Seen',
    close=False,
)
display(fig_ks)

fig_w1 = _plot_metric(
    curves,
    metric='w1_distance',
    lo='w1_distance_p10',
    hi='w1_distance_p90',
    out_path=str(OUT_DIR / 'w1_distance_vs_images_seen.pdf'),
    title='W1 Distance vs Images Seen',
    close=False,
)
display(fig_w1)

for fig in _plot_train_val(df, str(OUT_DIR), close=False):
    display(fig)

plt.close('all')
print(f'Wrote plots to {OUT_DIR}')

In [ ]:
import numpy as np
import matplotlib.colors as mcolors
from scripts.analyze_exchangeability import (
    _resolve_run_id,
    _list_width_dirs,
    _list_group_dirs,
    _collect_target_steps,
    _missing_weight_artifacts,
    _extract_weights_from_artifacts,
    _weight_similarity_matrix,
    _collect_member_states,
    _activation_similarity_matrix,
    _build_probe_loader,
)
from src.experiment.exchangeability_utils import (
    build_member_ids,
    extract_across_values,
    extract_within_values,
    shuffled_similarity_values_batched,
)

ECDF_RUN_ID = 'exchangeability'
ECDF_RUN_ID_RESOLUTION = 'latest_prefix'
ECDF_WIDTH = int(sorted(df['width'].astype(int).unique().tolist())[0])
ECDF_REPRESENTATION = 'weights'  # 'weights' or 'activations'
ECDF_SHUFFLE_REPEATS = 64
ECDF_SHUFFLE_BATCH_SIZE = 8
ECDF_MAX_POINTS = 200000
ECDF_STEPS = None  # set list[int] to restrict specific P values
ECDF_RNG_SEED = 0

# Used only when ECDF_REPRESENTATION='activations'
ECDF_PROBE_BATCH_SIZE = 1024
ECDF_PROBE_LOADER_BATCH_SIZE = 128
ECDF_PROBE_SEED = 1234

In [ ]:
def _ecdf_xy(values: np.ndarray):
    xs = np.sort(values)
    ys = np.arange(1, xs.size + 1, dtype=np.float64) / float(xs.size)
    return xs, ys


def _subsample(values: np.ndarray, max_points: int, rng: np.random.Generator):
    if max_points <= 0 or values.size <= max_points:
        return values
    idx = rng.choice(values.size, size=max_points, replace=False)
    return values[idx]


def _lighten(color, amount: float):
    rgba = np.array(mcolors.to_rgba(color), dtype=np.float64)
    white = np.array([1.0, 1.0, 1.0, rgba[3]], dtype=np.float64)
    mixed = (1.0 - amount) * rgba + amount * white
    return tuple(mixed.tolist())


resolved_run_id = _resolve_run_id(str(BASE_SAVE_DIR), ECDF_RUN_ID, ECDF_RUN_ID_RESOLUTION)
width_dirs = _list_width_dirs(str(BASE_SAVE_DIR), resolved_run_id)
if ECDF_WIDTH not in width_dirs:
    raise ValueError(f'Width {ECDF_WIDTH} not found under run {resolved_run_id}. Available widths: {sorted(width_dirs.keys())}')

group_dirs = _list_group_dirs(width_dirs[ECDF_WIDTH])
if not group_dirs:
    raise ValueError(f'No group directories found for width {ECDF_WIDTH}.')

target_steps = _collect_target_steps(group_dirs)
if ECDF_STEPS is None:
    steps = target_steps
else:
    target_set = set(target_steps)
    steps = [int(s) for s in ECDF_STEPS if int(s) in target_set]

if not steps:
    raise ValueError('No valid P values were selected for ECDF plotting.')

probe_loader = None
if ECDF_REPRESENTATION == 'activations':
    probe_loader = _build_probe_loader(
        probe_batch_size=ECDF_PROBE_BATCH_SIZE,
        probe_seed=ECDF_PROBE_SEED,
        probe_loader_batch_size=ECDF_PROBE_LOADER_BATCH_SIZE,
    )

ecdf_data = {}
for step in steps:
    if ECDF_REPRESENTATION == 'weights':
        missing = _missing_weight_artifacts(group_dirs, step)
        if missing:
            print(f'Skipping P={step}: missing weight artifacts ({len(missing)} files).')
            continue
        weight_chunks = [_extract_weights_from_artifacts(group_dir, step) for group_dir in group_dirs]
        weights = np.concatenate(weight_chunks, axis=0)
        num_members = int(weights.shape[0])
        sim = _weight_similarity_matrix(weights)
    elif ECDF_REPRESENTATION == 'activations':
        member_states = _collect_member_states(group_dirs, step, progress_label=f'ecdf w{ECDF_WIDTH} p{step}')
        num_members = len(member_states)
        sim = _activation_similarity_matrix(
            member_states,
            ECDF_WIDTH,
            probe_loader,
            progress_label=f'ecdf w{ECDF_WIDTH} p{step}',
        )
    else:
        raise ValueError("ECDF_REPRESENTATION must be 'weights' or 'activations'.")

    member_ids = build_member_ids(num_members, ECDF_WIDTH)
    within_real = extract_within_values(sim, member_ids)
    across_real = extract_across_values(sim, member_ids)

    rng = np.random.default_rng(ECDF_RNG_SEED + int(step))
    within_shuffled_parts = []
    remaining = int(ECDF_SHUFFLE_REPEATS)
    while remaining > 0:
        batch_size = min(ECDF_SHUFFLE_BATCH_SIZE, remaining)
        _, within_batch = shuffled_similarity_values_batched(
            similarity_matrix=sim,
            num_members=num_members,
            width=ECDF_WIDTH,
            rng=rng,
            batch_size=batch_size,
        )
        within_shuffled_parts.append(within_batch.reshape(-1))
        remaining -= batch_size
    within_shuffled = np.concatenate(within_shuffled_parts)

    within_real = _subsample(within_real, ECDF_MAX_POINTS, rng)
    across_real = _subsample(across_real, ECDF_MAX_POINTS, rng)
    within_shuffled = _subsample(within_shuffled, ECDF_MAX_POINTS, rng)

    ecdf_data[int(step)] = {
        'within_real': within_real,
        'across_real': across_real,
        'within_shuffled': within_shuffled,
    }

print(f'Prepared ECDF distributions for {len(ecdf_data)} P values.')

In [ ]:
if not ecdf_data:
    raise ValueError('No ECDF data was prepared.')

steps_sorted = sorted(ecdf_data.keys())
fig, ax = plt.subplots(figsize=(12, 8))
cmap = plt.get_cmap('viridis')
den = max(len(steps_sorted) - 1, 1)

for idx, step in enumerate(steps_sorted):
    base_color = cmap(idx / den)
    pair_color = _lighten(base_color, amount=0.35)

    x_wr, y_wr = _ecdf_xy(ecdf_data[step]['within_real'])
    x_ar, y_ar = _ecdf_xy(ecdf_data[step]['across_real'])
    x_ws, y_ws = _ecdf_xy(ecdf_data[step]['within_shuffled'])

    ax.plot(x_wr, y_wr, color=base_color, linewidth=2.0, linestyle='-', label=f'P={step} within_real')
    ax.plot(x_ar, y_ar, color=pair_color, linewidth=1.7, linestyle='-', label=f'P={step} across_real')
    ax.plot(x_ws, y_ws, color=pair_color, linewidth=1.7, linestyle='--', label=f'P={step} within_shuffled')

ax.set_xlabel('Absolute cosine similarity')
ax.set_ylabel('ECDF')
ax.set_title(f'ECDF by P (width={ECDF_WIDTH}, representation={ECDF_REPRESENTATION})')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=7, ncol=3)
fig.tight_layout()

ecdf_out = OUT_DIR / f'ecdf_similarity_by_p_w{ECDF_WIDTH}_{ECDF_REPRESENTATION}.pdf'
fig.savefig(ecdf_out, bbox_inches='tight')
display(fig)
plt.close(fig)
print(f'Wrote {ecdf_out}')